# Phase 4: features + Ridge

Models that *learn* the rule from train (2016-2023), scored on 2024 val. Bar to beat: carry-over, weighted val MAE 0.231. Run `.venv/bin/python -m src.data.load` once first if `data/raw/` is empty.

In [1]:
import sys
from pathlib import Path

# Let the notebook import code from src/ (the notebook lives one folder down).
sys.path.insert(0, str(Path.cwd().parent))

import polars as pl
from src.data.load import load_pbp
from src.features.qb_games import qb_games
from src.features.target import build_targets
from src.features.history import BASE_FEATURES, base_features
from src.models.baselines import carry_over, league_average
from src.models.linear import fit_linear, predict_linear
from src.models.evaluate import bootstrap_compare, mae

pbp = load_pbp(list(range(2016, 2026)))
games = qb_games(pbp)           # history: every QB-game, any size
targets = build_targets(games)  # what we predict and score
league_avg = league_average(targets.filter(pl.col("split") == "train"))

## Concept 1: linear regression learns the blend

Carry-over mixes this season and last season with a hand-picked K=300. Linear regression gets the same ingredients and learns one weight per feature from train: `prediction = w0 + w1*std_avg + w2*std_dropbacks + w3*last_avg + w4*last_dropbacks`.

In [2]:
feats = base_features(games, targets, league_avg)
train = feats.filter(pl.col("split") == "train")
val = feats.filter(pl.col("split") == "val")

model = fit_linear(train, BASE_FEATURES)
print(f"intercept {model.intercept_:+.4f}")
for name, w in zip(BASE_FEATURES, model.coef_):
    print(f"{name:15s} {w:+.6f}")

val_pred = predict_linear(model, val, BASE_FEATURES)
carry = carry_over(games, val, league_avg, k=300).select("passer_id", "season", "week", pl.col("prediction").alias("carry_over"))
val_all = val_pred.join(carry, on=["passer_id", "season", "week"])
print("\nlinear     val MAE", mae(val_pred))
print("carry_over val MAE", mae(val_all, "carry_over"))
print("bootstrap  ", bootstrap_compare(val_all, "carry_over", "prediction"))

intercept -0.0215
std_avg         +0.202406
std_dropbacks   +0.000071
last_avg        +0.165275
last_dropbacks  +0.000098

linear     val MAE {'weighted': 0.23089889670731564, 'unweighted': 0.25996496360583543}
carry_over val MAE {'weighted': 0.23116817069644627, 'unweighted': 0.2608993333893051}
bootstrap   {'gain': 0.0002692739891306036, 'win_rate': 0.5452, 'low': -0.005723974487672697, 'high': 0.006171402224602971}


### ▶ Play: Concept 1
Choose which ingredients the model gets and see how the learned weights and val MAE change.

In [3]:
# ---- KNOBS -----------------------------------------------------------------
FEATURES = ["std_avg", "last_avg"]  # Try ["std_avg"] alone, or add "std_dropbacks" / "last_dropbacks" back.
WEEK = 8                            # Which 2024 week to show example QBs from.
# ---------------------------------------------------------------------------

m = fit_linear(train, FEATURES)
p = predict_linear(m, val, FEATURES)
print("weights:", {n: round(float(w), 4) for n, w in zip(FEATURES, m.coef_)}, "intercept", round(float(m.intercept_), 4))
print(f"val MAE  default (all 4): {mae(val_pred)['weighted']:.4f}   now: {mae(p)['weighted']:.4f}")
show = ["L.Jackson", "J.Allen", "B.Nix", "H.Hooker"]
print(p.filter(pl.col("week") == WEEK, pl.col("passer").is_in(show))
       .join(val_pred.select("passer_id", "week", pl.col("prediction").alias("default_pred")), on=["passer_id", "week"])
       .select("passer", "std_avg", "std_dropbacks", "last_avg", "default_pred", "prediction", "target"))

weights: {'std_avg': 0.2282, 'last_avg': 0.2212} intercept 0.0262
val MAE  default (all 4): 0.2309   now: 0.2351
shape: (4, 7)
┌───────────┬───────────┬───────────────┬──────────┬──────────────┬────────────┬───────────┐
│ passer    ┆ std_avg   ┆ std_dropbacks ┆ last_avg ┆ default_pred ┆ prediction ┆ target    │
│ ---       ┆ ---       ┆ ---           ┆ ---      ┆ ---          ┆ ---        ┆ ---       │
│ str       ┆ f64       ┆ u32           ┆ f64      ┆ f64          ┆ f64        ┆ f64       │
╞═══════════╪═══════════╪═══════════════╪══════════╪══════════════╪════════════╪═══════════╡
│ L.Jackson ┆ 0.356163  ┆ 229           ┆ 0.121667 ┆ 0.149178     ┆ 0.134407   ┆ 0.250482  │
│ J.Allen   ┆ 0.309913  ┆ 218           ┆ 0.199235 ┆ 0.161239     ┆ 0.141011   ┆ 0.325827  │
│ B.Nix     ┆ -0.104731 ┆ 259           ┆ 0.05556  ┆ -0.015079    ┆ 0.014618   ┆ 0.335967  │
│ H.Hooker  ┆ -1.117706 ┆ 3             ┆ 0.05556  ┆ -0.238364    ┆ -0.21652   ┆ -0.782364 │
└───────────┴───────────┴───────────

## Concept 2: pre-shrunk features

Give the model the shrunk averages (K=250 imaginary league-average dropbacks) instead of the raw ones, so one fixed weight fits a 3-dropback QB and a 300-dropback QB alike.

In [4]:
from src.features.history import SHRUNK_FEATURES, shrunk_features

feats2 = shrunk_features(games, targets, league_avg)
train2 = feats2.filter(pl.col("split") == "train")
val2 = feats2.filter(pl.col("split") == "val")

model2 = fit_linear(train2, SHRUNK_FEATURES)
print(f"intercept {model2.intercept_:+.4f}", {n: round(float(w), 3) for n, w in zip(SHRUNK_FEATURES, model2.coef_)})
val_pred2 = predict_linear(model2, val2, SHRUNK_FEATURES)
val_all2 = val_pred2.join(carry, on=["passer_id", "season", "week"])
print("shrunk linear val MAE", mae(val_pred2))
print("bootstrap vs carry-over", bootstrap_compare(val_all2, "carry_over", "prediction"))

# Where does it lose? Error by part of the season (positive = carry-over better).
w = pl.col("weight").cast(pl.Float64)
err = lambda c: ((pl.col("target") - pl.col(c)).abs() * w).sum() / w.sum()
print(val_all2.with_columns(pl.when(pl.col("week") <= 4).then(pl.lit("1-4")).when(pl.col("week") <= 9).then(pl.lit("5-9"))
                            .otherwise(pl.lit("10+")).alias("weeks"))
      .group_by("weeks").agg(err("carry_over").alias("carry_over"), err("prediction").alias("shrunk_linear"))
      .with_columns((pl.col("shrunk_linear") - pl.col("carry_over")).alias("carry_over_better_by")).sort("weeks"))

intercept -0.0411 {'std_shrunk': 0.863, 'last_shrunk': 0.506}
shrunk linear val MAE {'weighted': 0.23282450028955481, 'unweighted': 0.2628308716666754}
bootstrap vs carry-over {'gain': -0.0016563295931085497, 'win_rate': 0.0504, 'low': -0.003513149114659145, 'high': 0.0003281032040497404}
shape: (3, 4)
┌───────┬────────────┬───────────────┬──────────────────────┐
│ weeks ┆ carry_over ┆ shrunk_linear ┆ carry_over_better_by │
│ ---   ┆ ---        ┆ ---           ┆ ---                  │
│ str   ┆ f64        ┆ f64           ┆ f64                  │
╞═══════╪════════════╪═══════════════╪══════════════════════╡
│ 1-4   ┆ 0.2378     ┆ 0.236845      ┆ -0.000954            │
│ 10+   ┆ 0.233019   ┆ 0.234473      ┆ 0.001455             │
│ 5-9   ┆ 0.22198    ┆ 0.226206      ┆ 0.004225             │
└───────┴────────────┴───────────────┴──────────────────────┘


### ▶ Play: Concept 2
Change how hard the features are shrunk.

In [5]:
# ---- KNOBS -----------------------------------------------------------------
K = 250   # Try 0 (raw averages again), 50, 600, 2000. Where does val MAE bottom out?
# ---------------------------------------------------------------------------

f = shrunk_features(games, targets, league_avg, k=K)
tr, va = f.filter(pl.col("split") == "train"), f.filter(pl.col("split") == "val")
m = fit_linear(tr, SHRUNK_FEATURES)
print("weights:", {n: round(float(x), 3) for n, x in zip(SHRUNK_FEATURES, m.coef_)})
print(f"val MAE  default (K=250): {mae(val_pred2)['weighted']:.4f}   now (K={K}): {mae(predict_linear(m, va, SHRUNK_FEATURES))['weighted']:.4f}")

weights: {'std_shrunk': 0.863, 'last_shrunk': 0.506}
val MAE  default (K=250): 0.2328   now (K=250): 0.2328


## Concept 3: interaction features (the handoff between seasons)

Carry-over split into two pieces: `last_piece = last season (shrunk) x dimmer` and `this_piece = this season x (1 - dimmer)`, with `dimmer = 300 / (dropbacks so far + 300)`. Weights of 1 and 1 give back carry-over exactly; the model learns its own.

In [6]:
from src.features.history import HANDOFF_FEATURES, handoff_features

key = ["passer_id", "season", "week"]
feats3 = handoff_features(games, targets, league_avg)
train3 = feats3.filter(pl.col("split") == "train")
val3 = feats3.filter(pl.col("split") == "val")
carry_all = carry_over(games, targets, league_avg, k=300).select(*key, pl.col("prediction").alias("carry_over"))

model3 = fit_linear(train3, HANDOFF_FEATURES)
print(f"intercept {model3.intercept_:+.4f}", {n: round(float(w), 3) for n, w in zip(HANDOFF_FEATURES, model3.coef_)}, "(carry-over = 0, 1, 1)")
val_all3 = predict_linear(model3, val3, HANDOFF_FEATURES).join(carry_all, on=key)
train_all3 = predict_linear(model3, train3, HANDOFF_FEATURES).join(carry_all, on=key)
print(f"val   MAE  handoff {mae(val_all3)['weighted']:.4f}   carry-over {mae(val_all3, 'carry_over')['weighted']:.4f}")
print(f"train MAE  handoff {mae(train_all3)['weighted']:.4f}   carry-over {mae(train_all3, 'carry_over')['weighted']:.4f}")
print("bootstrap vs carry-over", bootstrap_compare(val_all3, "carry_over", "prediction"))

intercept -0.0147 {'last_piece': 0.829, 'this_piece': 1.027} (carry-over = 0, 1, 1)
val   MAE  handoff 0.2322   carry-over 0.2312
train MAE  handoff 0.2261   carry-over 0.2263
bootstrap vs carry-over {'gain': -0.0010005797011287771, 'win_rate': 0.1314, 'low': -0.002539482525026756, 'high': 0.000720821772559216}


### ▶ Play: Concept 3
Change how fast the dimmer fades last season out.

In [7]:
# ---- KNOBS -----------------------------------------------------------------
K = 300   # Try 100 (last season fades fast) or 1000 (it lingers all year). Do the learned weights move toward 1?
# ---------------------------------------------------------------------------

f = handoff_features(games, targets, league_avg, k=K)
tr, va = f.filter(pl.col("split") == "train"), f.filter(pl.col("split") == "val")
m = fit_linear(tr, HANDOFF_FEATURES)
print("weights:", {n: round(float(x), 3) for n, x in zip(HANDOFF_FEATURES, m.coef_)}, "intercept", round(float(m.intercept_), 4))
print(f"val MAE  default (K=300): {mae(val_all3)['weighted']:.4f}   now (K={K}): {mae(predict_linear(m, va, HANDOFF_FEATURES))['weighted']:.4f}")

weights: {'last_piece': 0.829, 'this_piece': 1.027} intercept -0.0147
val MAE  default (K=300): 0.2322   now (K=300): 0.2322


## Concept 4: opponent defense strength

`def_shrunk` = EPA/dropback the opponent's defense allowed in weeks < N this season, shrunk toward the league average (K=200, picked on train). Model: linear regression on carry-over's prediction + def_shrunk.

In [8]:
from src.features.opponent import DEFENSE_FEATURES, defense_features

F4 = ["carry_over", *DEFENSE_FEATURES]
feats4 = defense_features(games, targets, league_avg, k=200).join(carry_all, on=key)
train4 = feats4.filter(pl.col("split") == "train")
val4 = feats4.filter(pl.col("split") == "val")
model4 = fit_linear(train4, F4)
print(f"intercept {model4.intercept_:+.4f}", {n: round(float(w), 3) for n, w in zip(F4, model4.coef_)})
val_all4 = predict_linear(model4, val4, F4)
print(f"val   MAE  +defense {mae(val_all4)['weighted']:.4f}   carry-over {mae(val_all4, 'carry_over')['weighted']:.4f}")
print(f"train MAE  +defense {mae(predict_linear(model4, train4, F4))['weighted']:.4f}   carry-over {mae(train4, 'carry_over')['weighted']:.4f}")
print("bootstrap vs carry-over", bootstrap_compare(val_all4, "carry_over", "prediction"))

# Is 2024 typical? Hold out each season in turn, fit on the other train seasons.
print("\nheld-out season: defense gain over a linear refit of carry-over alone (positive = defense helps)")
for s in range(2016, 2025):
    tr = feats4.filter(pl.col("split") == "train", pl.col("season") != s)
    te = feats4.filter(pl.col("season") == s)
    with_def = mae(predict_linear(fit_linear(tr, F4), te, F4))["weighted"]
    without = mae(predict_linear(fit_linear(tr, ["carry_over"]), te, ["carry_over"]))["weighted"]
    print(f"  {s}  {without - with_def:+.4f}")

intercept -0.0484 {'carry_over': 0.978, 'def_shrunk': 0.5}
val   MAE  +defense 0.2327   carry-over 0.2312
train MAE  +defense 0.2245   carry-over 0.2263
bootstrap vs carry-over {'gain': -0.001495893454784167, 'win_rate': 0.204, 'low': -0.004773196477558932, 'high': 0.002067182430901367}

held-out season: defense gain over a linear refit of carry-over alone (positive = defense helps)
  2016  +0.0032
  2017  +0.0006
  2018  -0.0012
  2019  +0.0041
  2020  +0.0048
  2021  +0.0005
  2022  -0.0001
  2023  +0.0011
  2024  -0.0003


### ▶ Play: Concept 4
Change how hard early-season defense numbers are shrunk.

In [9]:
# ---- KNOBS -----------------------------------------------------------------
K = 200   # Try 0 (trust week 1's defense number fully) or 2000 (barely move off average). Watch the defense weight.
# ---------------------------------------------------------------------------

f = defense_features(games, targets, league_avg, k=K).join(carry_all, on=key)
tr, va = f.filter(pl.col("split") == "train"), f.filter(pl.col("split") == "val")
m = fit_linear(tr, F4)
print("weights:", {n: round(float(x), 3) for n, x in zip(F4, m.coef_)})
print(f"train MAE  now (K={K}): {mae(predict_linear(m, tr, F4))['weighted']:.4f}")
print(f"val   MAE  default (K=200): {mae(val_all4)['weighted']:.4f}   now (K={K}): {mae(predict_linear(m, va, F4))['weighted']:.4f}")

weights: {'carry_over': 0.978, 'def_shrunk': 0.5}
train MAE  now (K=200): 0.2245
val   MAE  default (K=200): 0.2327   now (K=200): 0.2327


## Protocol v2: walk-forward re-run of Concepts 1-4

Folds 2019-2024, each trained only on earlier seasons (league average and every K re-picked per fold). Residual models: carry-over's weight fixed at 1. Keep = pooled bootstrap win ≥ 80% AND better in ≥ 4 of 6 seasons, judged in order against the current best. 2025 is dropped before anything runs (see DECISIONS.md).

In [10]:
from src.models.walk_forward import Experiment, compare, walk_forward

DEF_K_GRID = (50, 100, 200, 300, 500, 800, 1200, 2000)
SHRINK_K_GRID = (100, 150, 200, 250, 300, 400, 600)
experiments = {
    "carry-over": Experiment("carry-over"),
    "control: constant shift": Experiment("control", ["zero"], lambda g, t, la, k: t.with_columns(pl.lit(0.0).alias("zero"))),
    "C1 raw features": Experiment("C1", BASE_FEATURES, lambda g, t, la, k: base_features(g, t, la)),
    "C2 shrunk features": Experiment("C2", SHRUNK_FEATURES, lambda g, t, la, k: shrunk_features(g, t, la, k), SHRINK_K_GRID),
    "C3 handoff pieces": Experiment("C3", HANDOFF_FEATURES, lambda g, t, la, k: handoff_features(g, t, la, k), SHRINK_K_GRID),
    "C4 C1 + opponent defense": Experiment("C4", BASE_FEATURES + DEFENSE_FEATURES,
                                           lambda g, t, la, k: defense_features(g, base_features(g, t, la), la, k), DEF_K_GRID),
}
pooled = {name: walk_forward(games, targets, exp)[0] for name, exp in experiments.items()}

# Judged in order: each candidate vs whatever was the current best when it ran.
order = [("control: constant shift", "carry-over"), ("C1 raw features", "carry-over"),
         ("C2 shrunk features", "C1 raw features"), ("C3 handoff pieces", "C1 raw features"),
         ("C4 C1 + opponent defense", "C1 raw features")]
rows = [{"experiment": "carry-over", "vs": "", "pooled MAE": round(mae(pooled["carry-over"])["weighted"], 4)}]
for name, vs in order:
    s = compare(pooled[name], pooled[vs])
    rows.append({"experiment": name, "vs": vs, "pooled MAE": round(s["mae"], 4), "vs MAE": round(s["best_mae"], 4),
                 "win %": round(100 * s["win_rate"], 1), "seasons won": f"{s['seasons_won']}/6",
                 **{str(y): round(g, 4) for y, g in s["per_season_gain"].items()},
                 "keep": s["keep"]})
pl.Config.set_tbl_cols(20)
scoreboard = pl.DataFrame(rows)
scoreboard

experiment,vs,pooled MAE,vs MAE,win %,seasons won,2019,2020,2021,2022,2023,2024,keep
str,str,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,bool
"""carry-over""","""""",0.2287,null,null,null,null,null,null,null,null,null,null
"""control: constant shift""","""carry-over""",0.2285,0.2287,74.4,"""4/6""",0.0002,-0.0018,0.0012,0.0021,0.0006,-0.0012,false
"""C1 raw features""","""carry-over""",0.2276,0.2287,98.9,"""5/6""",0.001,-0.0014,0.0013,0.0026,0.0021,0.0005,true
"""C2 shrunk features""","""C1 raw features""",0.2286,0.2276,0.4,"""0/6""",-0.0009,-0.0007,-0.0008,-0.0001,-0.0014,-0.0017,false
"""C3 handoff pieces""","""C1 raw features""",0.2286,0.2276,0.4,"""1/6""",-0.0008,-0.0011,-0.0011,0.0002,-0.0014,-0.0015,false
"""C4 C1 + opponent defense""","""C1 raw features""",0.226,0.2276,99.8,"""4/6""",0.0038,0.0044,0.0005,-0.0001,0.0012,-0.0002,true


## Your exploration

Add cells below.